# Bori SLM Training Pipeline

**Save & Run All ready.** Close your laptop — this runs unattended.

| Phase | Est. Time | Steps |
|-|-|-|
| Pre-training | ~6h | 5,000 |
| SFT | ~2h | 1,500 |
| **Total** | **~8h** | *within 12h limit* |

Track live progress at [wandb.ai](https://wandb.ai) → `bori-slm` project.


### Step 0: Setup Codebase

In [ ]:
import os, shutil
input_dir = "/kaggle/input/bori-code"
working_dir = "/kaggle/working"

# Clean up working dir to prevent stale state from previous manual runs
for item in os.listdir(working_dir):
    path = os.path.join(working_dir, item)
    if item != 'checkpoints' and item != 'sft_checkpoints':  # Keep checkpoints if any
        try:
            if os.path.isdir(path): shutil.rmtree(path)
            else: os.remove(path)
        except Exception as e:
            print(f'Warning: Could not clean {item}: {e}')

# Copy codebase from kaggle input if available
if os.path.exists(input_dir):
    print('Copying uploaded bori-code dataset to working directory...')
    subdirs = [d for d in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, d)) and not d.startswith('.')]
    src_path = os.path.join(input_dir, subdirs[0]) if len(subdirs) == 1 else input_dir
    for item in os.listdir(src_path):
        s = os.path.join(src_path, item)
        d = os.path.join(working_dir, item)
        if os.path.isdir(s): shutil.copytree(s, d, dirs_exist_ok=True)
        else: shutil.copy2(s, d)

os.makedirs("/kaggle/working/src", exist_ok=True)
os.makedirs("/kaggle/working/scripts", exist_ok=True)
os.makedirs("/kaggle/working/configs", exist_ok=True)

with open("/kaggle/working/src/data.py", "w", encoding="utf-8") as f:
    f.write('import torch\nfrom datasets import load_dataset, interleave_datasets\nfrom itertools import chain\n\ndef _load_with_split_fallback(path, split, streaming, **kwargs):\n    """Try loading with the requested split; if it doesn\'t exist, try common alternatives."""\n    try:\n        return load_dataset(path, split=split, streaming=streaming, **kwargs)\n    except ValueError as e:\n        if "Unknown split" not in str(e):\n            raise\n        # Try common training split names\n        for alt in ["train_sft", "train_gen", "train", "test_sft", "test"]:\n            if alt == split:\n                continue\n            try:\n                ds = load_dataset(path, split=alt, streaming=streaming, **kwargs)\n                print(f"  Split \'{split}\' not found → using \'{alt}\' instead.")\n                return ds\n            except (ValueError, Exception):\n                continue\n        raise  # Re-raise original if no alternative works\n\ndef _load_multiple_datasets(dataset_names, split, streaming, probabilities=None, seed=42):\n    if isinstance(dataset_names, str):\n        dataset_names = [d.strip() for d in dataset_names.split(\',\')]\n        \n    if isinstance(probabilities, str):\n        probabilities = [float(p.strip()) for p in probabilities.split(\',\')]\n        \n    datasets = []\n    for name in dataset_names:\n        name = name.strip()\n        filter_col = None\n        filter_val = None\n        \n        # Check for query filtering, e.g. path:config?col=val or path?col=val\n        if "?" in name:\n            name, query = name.split("?", 1)\n            if "=" in query:\n                filter_col, filter_val = query.split("=", 1)\n                filter_col = filter_col.strip()\n                filter_val = filter_val.strip()\n        \n        if ":" in name:\n            path, config = name.split(":", 1)\n            print(f"Loading dataset {path} with configuration \'{config}\'...")\n            ds = _load_with_split_fallback(path, split, streaming, name=config)\n        else:\n            print(f"Loading dataset {name}...")\n            ds = _load_with_split_fallback(name, split, streaming)\n            \n        if filter_col and filter_val:\n            print(f"Filtering dataset \'{name}\' where \'{filter_col}\' == \'{filter_val}\'...")\n            # Capture filter_col and filter_val correctly in lambda closure\n            ds = ds.filter(lambda x, col=filter_col, val=filter_val: str(x.get(col)) == val)\n            \n        datasets.append(ds)\n    \n    if len(datasets) == 1:\n        dataset = datasets[0]\n    else:\n        # Interleave allows streaming from multiple sources at once according to the probabilities\n        # Passing seed ensures deterministic interleaving order\n        dataset = interleave_datasets(datasets, probabilities=probabilities, seed=seed)\n    \n    if streaming:\n        # Shuffle the stream deterministically\n        dataset = dataset.shuffle(seed=seed, buffer_size=10000)\n    return dataset\n\ndef get_packed_dataset(tokenizer, dataset_name, text_column="text", max_seq_length=4096, split="train", streaming=True, probabilities=None, seed=42):\n    """\n    Loads one or more datasets and packs multiple short documents into a single max_seq_length context window.\n    This is highly efficient for pre-training.\n    """\n    dataset = _load_multiple_datasets(dataset_name, split, streaming, probabilities, seed=seed)\n    \n    # Auto-detect the correct text column in case it\'s named \'content\', \'body\', etc.\n    available_keys = []\n    try:\n        # Peek at the first sample in the stream to read its keys\n        sample_iter = iter(dataset)\n        first_sample = next(sample_iter)\n        available_keys = list(first_sample.keys())\n        if text_column not in available_keys:\n            for alt in ["content", "body", "markdown", "document"]:\n                if alt in available_keys:\n                    text_column = alt\n                    print(f"Auto-detected text column name: \'{text_column}\'")\n                    break\n            else:\n                # Fallback to first available key\n                text_column = available_keys[0]\n                print(f"Warning: Default text column not found. Falling back to: \'{text_column}\'")\n    except Exception:\n        # Fallback to default if stream peeking fails\n        pass\n    \n    def tokenize_function(examples):\n        return tokenizer(examples[text_column])\n\n    tokenized_dataset = dataset.map(\n        tokenize_function,\n        batched=True,\n        remove_columns=available_keys if available_keys else None,\n    )\n\n    def group_texts(examples):\n        # Concatenate all texts. Only process tokenizer-generated keys\n        # to avoid unpacking metadata columns like language_score (floats) or id (strings)\n        tokenizer_keys = [k for k in ["input_ids", "attention_mask", "token_type_ids"] if k in examples]\n        concatenated_examples = {k: list(chain(*examples[k])) for k in tokenizer_keys}\n        total_length = len(concatenated_examples[list(concatenated_examples.keys())[0]])\n        \n        # Drop the small remainder, we could add padding if the model supported it\n        # instead of this drop, you can customize this part to your needs.\n        if total_length >= max_seq_length:\n            total_length = (total_length // max_seq_length) * max_seq_length\n            \n        # Split by chunks of max_len.\n        result = {\n            k: [t[i : i + max_seq_length] for i in range(0, total_length, max_seq_length)]\n            for k, t in concatenated_examples.items()\n        }\n        result["labels"] = result["input_ids"].copy()\n        return result\n\n    packed_dataset = tokenized_dataset.map(\n        group_texts,\n        batched=True,\n    )\n    \n    return packed_dataset\n\ndef get_sft_dataset(dataset_name, tokenizer, split="train", max_seq_length=2048, probabilities=None):\n    """\n    Helper for formatting a conversational dataset (like ShareGPT, Alpaca, or standard Messages) for SFT.\n    Dynamically normalizes different structures to the standard chat template.\n    """\n    dataset = _load_multiple_datasets(dataset_name, split, streaming=False, probabilities=probabilities)\n    \n    def format_chat(example):\n        # 1. ShareGPT format detection (e.g. SlimOrca-Dedup)\n        if "conversations" in example and isinstance(example["conversations"], list):\n            messages = []\n            for msg in example["conversations"]:\n                role = msg.get("from", "user")\n                if role in ["human", "user"]:\n                    mapped_role = "user"\n                  elif role in ["gpt", "assistant", "chatgpt"]:\n                    mapped_role = "assistant"\n                  elif role in ["system"]:\n                    mapped_role = "system"\n                  else:\n                    mapped_role = "user"\n                  messages.append({"role": mapped_role, "content": msg.get("value", "")})\n        \n        # 2. Alpaca format detection (e.g. jojo0217/korean_safe_conversation)\n        elif "instruction" in example and "output" in example:\n            user_content = example["instruction"]\n            if "input" in example and example["input"]:\n                user_content += "\\n" + example["input"]\n            messages = [\n                {"role": "user", "content": user_content},\n                {"role": "assistant", "content": example["output"]}\n            ]\n            \n        # 3. Standard Hugging Face messages format detection\n        elif "messages" in example:\n            messages = example["messages"]\n            \n        else:\n            raise ValueError(f"Dataset formatting not recognized. Keys present: {list(example.keys())}. "\n                             "Please ensure your dataset follows Alpaca, ShareGPT, or standard ChatML formatting.")\n            \n        # Sanitize: ensure all message content is a string (some datasets have None)\n        messages = [{"role": m.get("role", "user"), "content": m.get("content") or ""} for m in messages]\n\n        # Apply the chat template defined in the tokenizer\n        example["text"] = tokenizer.apply_chat_template(\n            messages, \n            tokenize=False, \n            add_generation_prompt=False\n        )\n        return example\n        \n    dataset = dataset.map(format_chat)\n    return dataset\n')

with open("/kaggle/working/src/model.py", "w", encoding="utf-8") as f:
    f.write('import json\nimport torch\nfrom transformers import Qwen2Config, Qwen2ForCausalLM\nfrom rich.console import Console\n\nconsole = Console()\n\ndef initialize_model_from_config(config_path: str, device: str = "cpu", vocab_size: int = None):\n    """\n    Initializes a new model randomly from the configuration for pre-training.\n    """\n    with open(config_path, "r") as f:\n        config_dict = json.load(f)\n        \n    config = Qwen2Config.from_dict(config_dict)\n    if vocab_size is not None:\n        console.print(f"[bold yellow]Aligning model vocab_size with tokenizer: {vocab_size} (was {config.vocab_size})[/bold yellow]")\n        config.vocab_size = vocab_size\n    \n    # Enable Flash Attention 2 if on supported hardware (Ampere+). T4 doesn\'t support it,\n    # so we will rely on sdpa (Scaled Dot Product Attention) which PyTorch 2.0+ uses by default.\n    # config._attn_implementation = "sdpa"\n    \n    console.print("[bold yellow]Initializing model weights...[/bold yellow]")\n    model = Qwen2ForCausalLM(config)\n    \n    # Calculate parameters\n    total_params = sum(p.numel() for p in model.parameters())\n    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    \n    console.print(f"[bold green]Model initialized successfully![/bold green]")\n    console.print(f"Total Parameters: {total_params / 1e6:.2f}M")\n    console.print(f"Trainable Parameters: {trainable_params / 1e6:.2f}M")\n    \n    return model\n\ndef load_model_from_checkpoint(checkpoint_path: str, device_map="auto"):\n    """\n    Loads an already pre-trained model for Fine-Tuning or Inference.\n    """\n    model = Qwen2ForCausalLM.from_pretrained(\n        checkpoint_path, \n        device_map=device_map,\n        torch_dtype=torch.float16 # Use float16 for T4 compatibility\n    )\n    return model\n')

with open("/kaggle/working/scripts/pretrain.py", "w", encoding="utf-8") as f:
    f.write('import os\nimport argparse\nimport time\nimport torch\nfrom transformers import (\n    Trainer,\n    TrainingArguments,\n    TrainerCallback,\n    PreTrainedTokenizerFast,\n    DataCollatorForLanguageModeling\n)\n\nclass ProgressCallback(TrainerCallback):\n    """Logs progress %, ETA, and elapsed time to W&B for remote monitoring."""\n    def __init__(self, phase="pretrain"):\n        self.phase = phase\n        self.start_time = None\n\n    def on_train_begin(self, args, state, control, **kwargs):\n        self.start_time = time.time()\n\n    def on_log(self, args, state, control, logs=None, **kwargs):\n        if state.max_steps <= 0 or self.start_time is None:\n            return\n        elapsed = time.time() - self.start_time\n        progress = state.global_step / state.max_steps\n        if state.global_step > 0:\n            eta_seconds = elapsed / progress * (1 - progress)\n            eta_minutes = eta_seconds / 60\n        else:\n            eta_minutes = 0\n        extra = {\n            "progress_pct": round(progress * 100, 1),\n            "eta_minutes": round(eta_minutes, 1),\n            "elapsed_minutes": round(elapsed / 60, 1),\n            "phase": self.phase,\n        }\n        if logs is not None:\n            logs.update(extra)\n        print(f"[{self.phase}] Step {state.global_step}/{state.max_steps} "\n              f"({extra[\'progress_pct\']}%) | ETA: {extra[\'eta_minutes\']}min | "\n              f"Elapsed: {extra[\'elapsed_minutes\']}min")\nimport wandb\nimport sys\nsys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))\n\nfrom src.model import initialize_model_from_config\nfrom src.data import get_packed_dataset\n\ndef main(args):\n    # Initialize wandb — resume="allow" lets multiple Kaggle sessions\n    # continue logging into the same W&B run seamlessly.\n    # Only initialize W&B on the main DDP process (Rank 0) to avoid HTTP 409 collisions\n    is_main_process = int(os.environ.get("RANK", 0)) == 0\n    if args.wandb_project and is_main_process:\n        run_id = args.wandb_run_id or args.run_name\n        os.environ["WANDB_RUN_ID"] = run_id\n        os.environ["WANDB_RESUME"] = "allow"\n        print(f"Initializing W&B run with ID: {run_id} (resume=\'allow\')")\n        wandb.init(\n            project=args.wandb_project,\n            name=args.run_name,\n            id=run_id,\n            resume="allow",\n        )\n\n    # 1. Load Tokenizer\n    print(f"Loading tokenizer from {args.tokenizer_path}...")\n    tokenizer = PreTrainedTokenizerFast.from_pretrained(args.tokenizer_path)\n    \n    # 2. Load Model\n    print("Initializing model...")\n    model = initialize_model_from_config(args.config_path, vocab_size=len(tokenizer))\n    \n    # 3. Load and Pack Dataset\n    print(f"Loading dataset {args.dataset_name}...")\n    train_dataset = get_packed_dataset(\n        tokenizer=tokenizer,\n        dataset_name=args.dataset_name,\n        max_seq_length=args.max_seq_length,\n        streaming=True,\n        probabilities=args.dataset_probs,\n        seed=args.seed\n    )\n    \n    # 4. Data Collator\n    data_collator = DataCollatorForLanguageModeling(\n        tokenizer=tokenizer,\n        mlm=False # Causal LM, not Masked LM\n    )\n    \n    # 5. Training Arguments\n    training_args = TrainingArguments(\n        output_dir=args.output_dir,\n        do_train=True,\n        # Batch size and accumulation for Kaggle (adjust based on VRAM)\n        per_device_train_batch_size=args.batch_size,\n        gradient_accumulation_steps=args.grad_accum,\n        learning_rate=args.learning_rate,\n        weight_decay=0.01,\n        # Max steps because we stream the dataset\n        max_steps=args.max_steps,\n        logging_steps=10,\n        # Robust Checkpointing\n        save_steps=args.save_steps,\n        save_total_limit=3, # Keep only the last 3 checkpoints to save Kaggle disk space\n        fp16=True, # T4 supports fp16. bf16 is for Ampere+\n        gradient_checkpointing=True, # Essential: trades compute for VRAM on T4\n        report_to="wandb" if (args.wandb_project and is_main_process) else "none",\n        optim="adafactor",\n        lr_scheduler_type="cosine",\n        warmup_steps=args.warmup_steps,\n        # Enable DDP if accelerate launched this with multiple GPUs\n        ddp_find_unused_parameters=False,\n        seed=args.seed, # Set global model seed\n        data_seed=args.seed, # Set dataset shuffle seed\n    )\n    \n    # 6. Initialize Trainer\n    trainer = Trainer(\n        model=model,\n        args=training_args,\n        train_dataset=train_dataset,\n        data_collator=data_collator,\n        callbacks=[ProgressCallback(phase="pretrain")],\n    )\n    \n    # 7. Train\n    print("Starting Training...")\n    # Check if there is a checkpoint to resume from\n    resume_from_checkpoint = False\n    if os.path.exists(args.output_dir) and len(os.listdir(args.output_dir)) > 0:\n        resume_from_checkpoint = True\n        print(f"Found existing checkpoints in {args.output_dir}, resuming training...")\n        \n    trainer.train(resume_from_checkpoint=resume_from_checkpoint)\n    \n    # 8. Save Final Model\n    trainer.save_model(args.output_dir)\n    tokenizer.save_pretrained(args.output_dir)\n    print("Training Complete. Model Saved.")\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config_path", type=str, default="configs/qwen2_bori_config.json")\n    parser.add_argument("--tokenizer_path", type=str, required=True)\n    parser.add_argument("--dataset_name", type=str, required=True, help="Comma-separated list of dataset names")\n    parser.add_argument("--dataset_probs", type=str, default=None, help="Comma-separated list of probabilities for interleaving")\n    parser.add_argument("--output_dir", type=str, default="./checkpoints")\n    parser.add_argument("--wandb_project", type=str, default=None)\n    parser.add_argument("--run_name", type=str, default="bori-pretrain")\n    parser.add_argument("--batch_size", type=int, default=1)\n    parser.add_argument("--grad_accum", type=int, default=32)\n    parser.add_argument("--learning_rate", type=float, default=3e-4)\n    parser.add_argument("--max_steps", type=int, default=10000)\n    parser.add_argument("--save_steps", type=int, default=1000)\n    parser.add_argument("--warmup_steps", type=int, default=500)\n    parser.add_argument("--max_seq_length", type=int, default=2048)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--wandb_run_id", type=str, default=None,\n                        help="Fixed W&B run ID for resuming the same run across Kaggle sessions")\n    \n    args = parser.parse_args()\n    main(args)\n')

with open("/kaggle/working/scripts/sft.py", "w", encoding="utf-8") as f:
    f.write('import os\nimport argparse\nimport sys\nimport time\nimport torch\nfrom transformers import (\n    AutoModelForCausalLM,\n    PreTrainedTokenizerFast,\n    TrainingArguments,\n    Trainer,\n    TrainerCallback,\n    DataCollatorForLanguageModeling\n)\n\nclass ProgressCallback(TrainerCallback):\n    """Logs progress %, ETA, and elapsed time to W&B for remote monitoring."""\n    def __init__(self, phase="sft"):\n        self.phase = phase\n        self.start_time = None\n\n    def on_train_begin(self, args, state, control, **kwargs):\n        self.start_time = time.time()\n\n    def on_log(self, args, state, control, logs=None, **kwargs):\n        if state.max_steps <= 0 or self.start_time is None:\n            return\n        elapsed = time.time() - self.start_time\n        progress = state.global_step / state.max_steps\n        if state.global_step > 0:\n            eta_seconds = elapsed / progress * (1 - progress)\n            eta_minutes = eta_seconds / 60\n        else:\n            eta_minutes = 0\n        extra = {\n            "progress_pct": round(progress * 100, 1),\n            "eta_minutes": round(eta_minutes, 1),\n            "elapsed_minutes": round(elapsed / 60, 1),\n            "phase": self.phase,\n        }\n        if logs is not None:\n            logs.update(extra)\n        print(f"[{self.phase}] Step {state.global_step}/{state.max_steps} "\n              f"({extra[\'progress_pct\']}%) | ETA: {extra[\'eta_minutes\']}min | "\n              f"Elapsed: {extra[\'elapsed_minutes\']}min")\nimport wandb\nsys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))\n\nfrom src.data import get_sft_dataset\n\ndef main(args):\n    # Initialize wandb — resume="allow" lets multiple Kaggle sessions\n    # continue logging into the same W&B run seamlessly.\n    # Only initialize W&B on the main DDP process (Rank 0) to avoid HTTP 409 collisions\n    is_main_process = int(os.environ.get("RANK", 0)) == 0\n    if args.wandb_project and is_main_process:\n        run_id = args.wandb_run_id or args.run_name\n        os.environ["WANDB_RUN_ID"] = run_id\n        os.environ["WANDB_RESUME"] = "allow"\n        print(f"Initializing W&B run with ID: {run_id} (resume=\'allow\')")\n        wandb.init(\n            project=args.wandb_project,\n            name=args.run_name,\n            id=run_id,\n            resume="allow",\n        )\n\n    print("Loading Tokenizer...")\n    tokenizer = PreTrainedTokenizerFast.from_pretrained(args.tokenizer_path)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    \n    print("Loading Pre-trained Model...")\n    model = AutoModelForCausalLM.from_pretrained(\n        args.model_path,\n        torch_dtype=torch.float32,\n    )\n    \n    print("Loading SFT Dataset...")\n    dataset = get_sft_dataset(\n        args.dataset_name, tokenizer, split="train",\n        max_seq_length=args.max_seq_length, probabilities=args.dataset_probs\n    )\n    \n    # Tokenize the formatted text\n    def tokenize_fn(examples):\n        return tokenizer(\n            examples["text"],\n            truncation=True,\n            max_length=args.max_seq_length,\n            padding=False,\n        )\n    \n    tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)\n    \n    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)\n    \n    training_args = TrainingArguments(\n        output_dir=args.output_dir,\n        per_device_train_batch_size=args.batch_size,\n        gradient_accumulation_steps=args.grad_accum,\n        learning_rate=args.learning_rate,\n        logging_steps=10,\n        num_train_epochs=args.epochs,\n        max_steps=args.max_steps,\n        save_strategy="steps" if args.max_steps > 0 else "epoch",\n        save_steps=args.save_steps if args.max_steps > 0 else None,\n        fp16=True,\n        gradient_checkpointing=True,\n        report_to="wandb" if (args.wandb_project and is_main_process) else "none",\n        optim="adafactor",\n        lr_scheduler_type="cosine",\n        warmup_ratio=0.1,\n        ddp_find_unused_parameters=False,\n    )\n    \n    print("Initializing Trainer...")\n    trainer = Trainer(\n        model=model,\n        train_dataset=tokenized,\n        args=training_args,\n        data_collator=collator,\n        callbacks=[ProgressCallback(phase="sft")],\n    )\n    \n    print("Starting SFT Training...")\n    resume_from_checkpoint = False\n    if os.path.exists(args.output_dir) and len(os.listdir(args.output_dir)) > 0:\n        resume_from_checkpoint = True\n        print(f"Found existing checkpoints in {args.output_dir}, resuming SFT training...")\n        \n    trainer.train(resume_from_checkpoint=resume_from_checkpoint)\n    \n    trainer.save_model(args.output_dir)\n    tokenizer.save_pretrained(args.output_dir)\n    print("SFT Training Complete.")\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--model_path", type=str, required=True, help="Path to pre-trained model")\n    parser.add_argument("--tokenizer_path", type=str, required=True)\n    parser.add_argument("--dataset_name", type=str, required=True)\n    parser.add_argument("--dataset_probs", type=str, default=None)\n    parser.add_argument("--output_dir", type=str, default="./sft_checkpoints")\n    parser.add_argument("--wandb_project", type=str, default=None)\n    parser.add_argument("--run_name", type=str, default="bori-sft")\n    parser.add_argument("--batch_size", type=int, default=1)\n    parser.add_argument("--grad_accum", type=int, default=32)\n    parser.add_argument("--learning_rate", type=float, default=2e-5)\n    parser.add_argument("--epochs", type=int, default=3)\n    parser.add_argument("--max_seq_length", type=int, default=2048)\n    parser.add_argument("--max_steps", type=int, default=-1)\n    parser.add_argument("--save_steps", type=int, default=500)\n    parser.add_argument("--wandb_run_id", type=str, default=None,\n                        help="Fixed W&B run ID for resuming the same run across Kaggle sessions")\n    \n    args = parser.parse_args()\n    main(args)\n')

with open("/kaggle/working/scripts/test_inference.py", "w", encoding="utf-8") as f:
    f.write('import argparse\nimport torch\nimport sys\nimport os\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--model_path", type=str, required=True)\n    parser.add_argument("--tokenizer_path", type=str, required=True)\n    args = parser.parse_args()\n\n    print("Loading model and tokenizer...")\n    tokenizer = AutoTokenizer.from_pretrained(args.tokenizer_path)\n    model = AutoModelForCausalLM.from_pretrained(\n        args.model_path,\n        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,\n        device_map="auto"\n    )\n\n    prompts = [\n        "안녕하세요, 오늘 날씨가 어떤가요?",\n        "Explain how continuous pre-training works in simple terms.",\n        "Konglish is a mixture of Korean and English. For example, \'I am going to have a meeting in the morning\' can be said in Konglish as \'오늘 아침에 미팅이 있어요.\'"\n    ]\n\n    for p in prompts:\n        print(f"\\nPrompt: {p}")\n        inputs = tokenizer(p, return_tensors="pt").to(model.device)\n        with torch.no_grad():\n            outputs = model.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=0.7)\n        completion = tokenizer.decode(outputs[0], skip_special_tokens=True)\n        print(f"Response: {completion}")\n\nif __name__ == "__main__":\n    main()\n')

config_file = "/kaggle/working/configs/qwen2_bori_config.json"
if not os.path.exists(config_file):
    import json
    with open(config_file, "w", encoding="utf-8") as f:
        json.dump({'architectures': ['Qwen2ForCausalLM'], 'attention_dropout': 0.0, 'bos_token_id': 151643, 'eos_token_id': 151645, 'hidden_act': 'silu', 'hidden_size': 1536, 'initializer_range': 0.02, 'intermediate_size': 4096, 'max_position_embeddings': 32768, 'max_window_layers': 28, 'model_type': 'qwen2', 'num_attention_heads': 12, 'num_hidden_layers': 18, 'num_key_value_heads': 2, 'rms_norm_eps': 1e-06, 'rope_theta': 1000000.0, 'sliding_window': 4096, 'tie_word_embeddings': True, 'use_cache': False, 'use_sliding_window': False, 'vocab_size': 151936}, f, indent=4)

print("Setup complete:", os.listdir(working_dir))


In [ ]:
import sys
!{sys.executable} -m pip install -q transformers>=4.40.0 accelerate>=0.29.0 datasets>=2.18.0 wandb>=0.16.0 tokenizers>=0.15.0 rich tqdm gradio


### Step 1: W&B Login (auto-skips if secret missing)

In [ ]:
try:
    import wandb
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret('wandb_api_key'))
    WANDB_PROJECT = 'bori-slm'
    print('W&B logged in. Track at: https://wandb.ai')
except Exception as e:
    print(f'W&B skipped ({e}). Training continues without logging.')
    WANDB_PROJECT = None


### Step 2: Configure

In [ ]:
import torch, os
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
num_gpus = torch.cuda.device_count()
accelerate_config = 'configs/accelerate_config_multi.yaml' if num_gpus > 1 else 'configs/accelerate_config_single.yaml'
print(f'{num_gpus} GPU(s) detected.')

# Ensure config files exist in configs/ directory
os.makedirs('configs', exist_ok=True)
if not os.path.exists('configs/accelerate_config_single.yaml'):
    with open('configs/accelerate_config_single.yaml', 'w') as f:
        f.write("compute_environment: LOCAL_MACHINE\ndistributed_type: 'NO'\ndowncast_bf16: 'no'\ngpu_ids: all\nmachine_rank: 0\nnum_machines: 1\nnum_processes: 1\nrdzv_backend: static\nsame_network: true\ntargets: []\ntread_size: 1\nuse_cpu: false\n")
if not os.path.exists('configs/accelerate_config_multi.yaml'):
    with open('configs/accelerate_config_multi.yaml', 'w') as f:
        f.write("compute_environment: LOCAL_MACHINE\ndistributed_type: MULTI_GPU\ndowncast_bf16: 'no'\ngpu_ids: all\nmachine_rank: 0\nnum_machines: 1\nnum_processes: 2\nrdzv_backend: static\nsame_network: true\ntargets: []\ntread_size: 2\nuse_cpu: false\n")

# ── Version name for this training run ────────────────────────────────────
RUN_NAME = 'bori-0.7b-v1.3'

# ── W&B run IDs (fixed so all Kaggle sessions log to the SAME W&B run) ───
WANDB_RUN_ID_PRETRAIN = 'bori-0.7b-t4x2-v1-pretrain'
WANDB_RUN_ID_SFT      = 'bori-0.7b-v1.3-sft'

# ── Checkpoint resume ─────────────────────────────────────────────────────
CHECKPOINT_DATASET = None


### Step 2b: Restore Checkpoint (auto-skips on first run)

In [ ]:
import shutil, os
if CHECKPOINT_DATASET and os.path.exists(CHECKPOINT_DATASET):
    print(f'Restoring pre-training checkpoint from {CHECKPOINT_DATASET}...')
    os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
    checkpoint_found = False
    for root, dirs, files in os.walk(CHECKPOINT_DATASET):
        if 'trainer_state.json' in files:
            basename = os.path.basename(root)
            if basename.startswith('checkpoint-'):
                print(f'Found valid checkpoint folder: {basename} at {root}')
                target_dir = f'/kaggle/working/checkpoints/{basename}'
                if os.path.exists(target_dir): shutil.rmtree(target_dir)
                shutil.copytree(root, target_dir)
                args_bin = os.path.join(target_dir, 'training_args.bin')
                if os.path.exists(args_bin):
                    os.remove(args_bin)
                    print(f'  🗑️ Removed training_args.bin to prevent overriding new arguments')
                checkpoint_found = True
    if checkpoint_found:
        print('Checkpoint restore complete:', os.listdir('/kaggle/working/checkpoints'))
    else:
        shutil.copytree(CHECKPOINT_DATASET, '/kaggle/working/checkpoints', dirs_exist_ok=True)
        print('Fallback copy complete:', os.listdir('/kaggle/working/checkpoints'))
else:
    print('No checkpoint dataset set — training from scratch.')


### Step 3: Pre-training (~6h)

In [ ]:
import sys, os
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
python_path = sys.executable
wandb_flag    = f'--wandb_project {WANDB_PROJECT}' if WANDB_PROJECT else ''
wandb_id_flag = f'--wandb_run_id {WANDB_RUN_ID_PRETRAIN}' if WANDB_PROJECT else ''
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True {python_path} -m accelerate.commands.launch \
    --config_file {accelerate_config} scripts/pretrain.py \
    --config_path configs/qwen2_bori_config.json \
    --tokenizer_path "Qwen/Qwen2-0.5B" \
    --dataset_name "HuggingFaceFW/fineweb:sample-10BT,HuggingFaceFW/fineweb-2:kor_Hang?dump=CC-MAIN-2024-18" \
    --dataset_probs "0.5,0.5" \
    --batch_size 1 --grad_accum 32 --seed 42 \
    --max_steps 5000 --save_steps 1000 \
    --output_dir /kaggle/working/checkpoints \
    {wandb_flag} {wandb_id_flag} --run_name {RUN_NAME}-pretrain


### Step 4: SFT (~2h)

In [ ]:
import sys, os
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
if not os.path.exists('/kaggle/working/checkpoints/config.json'):
    raise ValueError('❌ ERROR: Pre-trained model config not found at /kaggle/working/checkpoints/config.json!')
python_path = sys.executable
wandb_flag    = f'--wandb_project {WANDB_PROJECT}' if WANDB_PROJECT else ''
wandb_id_flag = f'--wandb_run_id {WANDB_RUN_ID_SFT}' if WANDB_PROJECT else ''
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True {python_path} -m accelerate.commands.launch \
    --config_file {accelerate_config} scripts/sft.py \
    --model_path /kaggle/working/checkpoints \
    --tokenizer_path "Qwen/Qwen2-0.5B" \
    --dataset_name "HuggingFaceH4/ultrachat_200k,jojo0217/korean_safe_conversation" \
    --dataset_probs "0.5,0.5" \
    --max_steps 1500 --save_steps 500 \
    --output_dir /kaggle/working/sft_checkpoints \
    {wandb_flag} {wandb_id_flag} --run_name {RUN_NAME}-sft


### Step 5: Export

In [ ]:
import shutil, os
for d in ['checkpoints','sft_checkpoints']:
    p = f'/kaggle/working/{d}'
    if os.path.exists(p):
        shutil.make_archive(f'/kaggle/working/{d}_export','zip',p)
        print(f'Exported {d}')
print('\n✅ All done! Download from Output tab.')
